# Help Desk Ticket SLA and Volume Forecasting

This capstone uses `data/raw/issues.csv` for two integrated workstreams: **(1) retrospective comparison of elapsed resolution time with project-configured SLA references by recorded priority; and (2) separate 7- and 30-day daily Ticket-volume forecasting experiments for capacity-planning research.** The 30-day forecast is a daily path, not one monthly total. The primary analysis includes `issue_type == Ticket` and creation dates on/after 2016-01-01 UTC, matching the period described in the accompanying source documentation.

**Important source caveat:** the CSV contains creation dates before 2016 despite the documentation's stated range. Those records are profiled but excluded from the primary scope. Dates without scoped Ticket rows are treated as zero arrivals in the forecasting series under an explicit completeness assumption that has not been independently verified. The source ends in March 2023 and cannot establish current operational performance.

The inherited SLA references are project assumptions: Blocker/Highest → Critical (4h), High (8h), Medium (24h), Low/Lowest → Low (24h); `unknown` is excluded. The measured duration is calendar wall-clock time between creation and recorded resolution—not a verified business-hours SLA clock. Results are retrospective, not contractual compliance or a live warning model.

**Shared retrospective result across this notebook sequence:** among 16,735 eligible completed Tickets, 3,333 (19.9%) met the configured reference. By mapped priority: Critical 524/2,124 (24.7%); High 576/3,146 (18.3%); Medium 2,161/11,161 (19.4%); Low 72/304 (23.7%). Eligibility requires exact Ticket type, documented 2016+ scope, `Done`, known mapped priority, and valid nonnegative elapsed duration. Notebook 02 shows the detailed comparison; these same counts and caveats are the project-wide retrospective baseline.


# 05 | Ethical interpretation and subgroup audit

The retrospective reference baseline is 3,333/16,735 eligible Tickets met (19.9%); the notebook below reports variation by priority, year and masked project group. These are operational/descriptive slices, not protected-group fairness measurements. The single aggregate forecast series has no queue/site/shift segments, so forecast-allocation fairness cannot be evaluated from this file.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display
from src.issues_capstone import run_fairness_audit
result = run_fairness_audit()
print(result['protected_attribute_status'])
print('Project groups:', result['project_group_count'])
display(result['priority_groups'])
display(result['year_groups'])


not present/approved; protected-group fairness cannot be assessed
Project groups: 302


,mapped_priority,assessed,reference_met,reference_met_rate,median_elapsed_hours,reference_met_wilson95_low,reference_met_wilson95_high
0,Critical,2124,524,0.246704,73.493472,0.228839,0.265484
1,High,3146,576,0.183090,112.619444,0.169965,0.196988
2,Low,304,72,0.236842,170.996944,0.192520,0.287731
3,Medium,11161,2161,0.193621,171.545764,0.186396,0.201056


,created_year,assessed,reference_met,reference_met_rate,median_elapsed_hours,reference_met_wilson95_low,reference_met_wilson95_high
0,2016,534,118,0.220974,188.636111,0.187846,0.258088
1,2017,1,0,0.000000,1249.376667,0.000000,0.793451
2,2018,1,0,0.000000,3002.122846,0.000000,0.793451
3,2019,2562,428,0.167057,166.968735,0.153113,0.181998
4,2020,2952,566,0.191734,166.098402,0.177938,0.206332
5,2021,4276,823,0.192470,176.408292,0.180931,0.204560
6,2022,5313,1144,0.215321,121.844167,0.204476,0.226577
7,2023,1096,254,0.231752,74.512639,0.207734,0.257643


## Fairness data-gap statement

This is a deliberate scope statement, not an omission. `data/raw/issues.csv` carries no protected-attribute fields at all: there is no age, gender, ethnicity, disability, location-as-proxy, or any other demographic or legally protected attribute anywhere in the source. Rubric expectations for a fairness/bias audit (demographic parity, equalized odds, protected-group disparate-impact testing) assume such attributes exist and can be joined to outcomes. That structural precondition is absent here, so those specific tests cannot be run — not because they were skipped, but because the data required to run them does not exist in this source.

Rather than skip the audit outright, this notebook substitutes the closest available **operational equity proxy**: the internal, non-demographic `project` field, together with `issue_priority` and issue-type breakdowns already used elsewhere in this project. These slices can still surface *operationally* uneven treatment (for example, one project's tickets being systematically slower to resolve or under-forecast relative to others), which is a legitimate and useful audit even though it is not a demographic fairness audit. The two must not be conflated: a clean result on the project-group proxy is evidence about operational consistency across an internal grouping, not evidence that the underlying process treats any protected group fairly.

**Forward-looking recommendation.** Before this project (or a production system built on it) is used to make decisions that affect people, the organization should establish a governed process to collect the minimum protected attributes needed for a real fairness audit — collected lawfully, with clear purpose limitation, access controls and a documented retention policy — and re-run demographic parity and equalized-odds checks against that data. Until that data exists, any claim of demographic fairness for this system would be unsupported by the available evidence.

## Auditing SLA outcomes by originating project

The `project` field is an internal, non-demographic grouping, but it is the closest thing this dataset has to an operational subgroup that could mask disparate treatment (for example, if tickets from one project are systematically deprioritized). This cell reads the pre-computed per-project SLA audit, sorts by the number of eligible (`assessed`) tickets per project, and shows the top 20 by volume alongside a `group_stability` tally. `group_stability` flags projects whose `assessed` count is too small for a stable met-rate estimate (a common fairness-audit caveat: a group with only a handful of tickets can show a misleadingly extreme met/missed rate purely by chance). Projects flagged as low-stability should not be used to claim a project is unusually well- or poorly-served without a larger sample.


In [2]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from src.issues_capstone import PROJECT_REPORTS
groups = pd.read_csv(PROJECT_REPORTS / "sla_group_audit_by_project.csv")
display(groups.sort_values("assessed", ascending=False).head(20))
display(groups["group_stability"].value_counts())


,issue_proj,assessed,reference_met_rate,group_stability
107,C06hg2xcu,1215,0.157202,at least 30 records
230,C12hgx18a,629,0.139905,at least 30 records
227,C12ggzcz0u,622,0.501608,at least 30 records
226,C12ggz_0,458,0.545852,at least 30 records
298,dx8a,400,0.202500,at least 30 records
241,C13ggb_0,368,0.084239,at least 30 records
108,C06hgfz_x,308,0.051948,at least 30 records
225,C12gg0.a,280,0.357143,at least 30 records
76,C06ggzcz0,248,0.290323,at least 30 records
54,C06gg10,224,0.379464,at least 30 records


group_stability
small group: unstable estimate    162
at least 30 records               140
Name: count, dtype: int64

## Safeguards and limitations

As stated in the fairness data-gap section above, no approved demographic attributes are present in this source, so demographic parity, equalized odds and protected-group fairness cannot be assessed here; the project-group audit above is an operational equity proxy, not a substitute for that assessment. Priority-specific reference rules make differences in attainment partly dependent on target construction. Small project groups are unstable and should not be ranked. Before operational use: validate coverage and policy, report signed forecast errors by approved service segments, keep human review, investigate persistent under-forecasting, and — before this system informs decisions that affect people — establish the governed protected-attribute data collection described above so a genuine demographic fairness audit can be run. The current aggregate history cannot support that segment-level forecast audit or any protected-group fairness claim.

## In summary

This notebook is the fairness and ethics check for the project. It states plainly that the data does not contain any personal or demographic information, like age, gender, or location, so a true fairness audit by protected group cannot be performed, not because it was skipped, but because that information was never collected. As the closest available substitute, the notebook compares outcomes across internal project groupings to look for uneven treatment, while being careful to note that this is a stand in, not a real demographic fairness test. It closes with a recommendation that if this project is ever used to affect real decisions, the organization should responsibly collect the right data first.